# Optimization of a Recycling Mixer-Reactor-Clarifier Activated Sludge System Using a Physically-Constrained Statistical Surrogate

This notebook is the staged executable companion to `article/wip_v2/manuscript.tex`. It uses the five-reactor ASM2d-TSN model, ten-layer Clarifier, 25-dimensional deterministic design, 170-response statistical surrogate, physical deployment QP, and five-dimensional bounded optimization documented there.

Every section is an independent gate. A failed gate stops execution; rejected mechanistic rows are retained for diagnosis and are never replaced.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

from closed_loop.workflow import ClosedLoopWorkflow

REPOSITORY_ROOT = Path.cwd().resolve()
CONFIG_PATH = REPOSITORY_ROOT / 'config' / 'params_closed_loop.json'
PROFILE = os.environ.get('CLOSED_LOOP_PROFILE', 'test_2000')
RUN_ID = os.environ.get('CLOSED_LOOP_RUN_ID')
if not RUN_ID:
    raise RuntimeError('Set CLOSED_LOOP_RUN_ID to a new immutable run identifier before execution.')

workflow = ClosedLoopWorkflow(
    config_path=CONFIG_PATH,
    profile=PROFILE,
    run_id=RUN_ID,
    repository_root=REPOSITORY_ROOT,
)
print(json.dumps({'profile': PROFILE, 'run_id': RUN_ID, 'run_root': str(workflow.run_root)}, indent=2))

## 1. Static mathematical and design contract

This gate reconstructs and audits the 28-by-20 stoichiometric matrix and five invariant rows, checks every configured dimension and coordinate order, generates the exact SplitMix64/Fisher–Yates Latin hypercube, and freezes the ordered development/assessment split.

In [ ]:
manifest = workflow.run(through='static')
manifest['stages']['static']

## 2. Mechanistic pilots and frequent generation checks

The first 256 immutable design rows are solved in checkpoints ending at 4, 16, 64, and 256 rows. Every checkpoint verifies the full steady-state, external-balance, positivity, Clarifier, and local-stability contract before more rows are attempted.

In [ ]:
manifest = workflow.run(through='pilot')
manifest['stages']['pilot']

## 3. Complete mechanistic design

Generation resumes from the sealed pilot chunks and proceeds through every configured checkpoint. For `test_2000`, this produces exactly 2,000 accepted 110-state steady solutions and their 170-coordinate targets. The `full` profile instead produces the article's independent 20,000-point design.

In [ ]:
manifest = workflow.run(through='dataset')
manifest['stages']['dataset']

## 4. Development fit and one-time assessment

The first 80% of rows determine every center, scale, feature, coefficient, and physical row scale. The remaining 20% are evaluated once as raw, equality-projected, and fully deployed predictions. Production fitting is prohibited unless the fixed predictive and correction-reliance gates pass.

In [ ]:
manifest = workflow.run(through='assessment')
manifest['stages']['assessment']

## 5. Production refit and physical replay

The unchanged estimator is refitted on every mechanistic row. All deployment QPs are replayed, effluent-quality scales and the maximum training leverage are frozen, and the production model is serialized only after the numerical checks pass.

In [ ]:
manifest = workflow.run(through='production')
manifest['stages']['production']

## 6. Nominal and robustness optimization

Each case uses boundary-augmented DIRECT and deterministic pattern refinement on the deployed surrogate, followed by selected-point mechanistic validation and an independent finite-budget mechanistic reference search. The verification profile reduces only case counts and search budgets; it executes the same branches as the full article procedure.

In [ ]:
manifest = workflow.run(through='optimization')
manifest['stages']['optimization']

## 7. Reports and immutable completion seal

Tables and figures are generated from row-level artifacts. Terminal acceptance verifies every stage marker and writes the completion seal; a sealed run identifier cannot be reused.

In [ ]:
manifest = workflow.run(through='report')
manifest = workflow.run(through='complete')
summary = {
    'run_id': manifest['run_id'],
    'profile': manifest['profile'],
    'article_eligible': manifest['article_eligible'],
    'status': manifest['status'],
    'completed': (workflow.run_root / 'COMPLETED.json').is_file(),
}
print(json.dumps(summary, indent=2))
summary